## 不使用 @tool 的方式定义工具

### 1. 举例

In [1]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [ ]:
# 2. 声明一个函数(工具)
def get_weather(city: str):
    '''获取天气的工具'''
    return f"{city}天气晴朗"

# 3. 将函数绑定在模型上
model_with_tools = model.bind_tools([get_weather])

# 4. 调用模型
response = model_with_tools.invoke("北京的天气怎么样")
rprint(response)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 34,
            'prompt_tokens': 154,
            'total_tokens': 188,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 22,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'glm-5.3',
        'system_fingerprint': None,
        'id': '20261110152910e05468f79d8a4cf6',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a1c45c-5ce0-76e1-b6e8-4bd5a815ec3b-0',
    tool_calls=[
        {'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_-7101058546476776019', 'type': 'tool_call'}
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 154,
        'output_tokens': 34,
        'total_tokens': 188,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 22}
    }
)

#### 2. 工具描述的各部分详解

##### 2.1 了解 convert_to_openai_tool

执行 model.bind_tools([get_weather]), 底层最终会调用 convert_to_openai_tool 生成工具描述. 所以我们可以直接调用后者查看解析后的工具描述.

In [3]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city: str):
    '''获取天气的工具'''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取天气的工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

#### 2.2 description 说明

In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city: str):
    '''获取天气的工具''' # description 字段内容
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取天气的工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

### 2.3 参数说明

convert_to_openai_tool 会从 docstring 加载参数说明, 这里的 docstring 必须遵循 Google 风格
Google 风格 docstring 说明: https://google.github.io/styleguide/pyguide.html

In [11]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city: str):
    '''
    获取天气的工具

    Args:
        city: 具体的城市

    Returns:
        返回城市的天气
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取天气的工具',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

#### 2.4 参数类型的说明

例1: 正确的

In [13]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city):
    '''
    获取天气的工具
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取天气的工具',
        'parameters': {'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}
    }
}

例2: 错误的

如下的代码运行会报错
要求: 如果在 docstring 中声明了参数的描述, 则必须在函数声明处指明参数的类型

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city):
    '''
    获取天气的工具

    Args:
        city: 具体的城市
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

#### 2.5 参数默认值的说明

一旦参数设置了默认值, 则打印的结果中的 required 字段中就不存在了

例1

In [16]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(city: str = "北京"):
    '''
    获取天气的工具

    Args:
        city: 具体的城市
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取天气的工具',
        'parameters': {
            'properties': {'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

例2

In [18]:
from langchain_core.utils.function_calling import convert_to_openai_tool
def get_weather(dt: str, city: str = "北京"):
    '''
    获取天气的工具

    Args:
        city: 具体的城市
        dt: 时间
    '''
    return f"{city}天气晴朗"

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取天气的工具',
        'parameters': {
            'properties': {
                'dt': {'description': '时间', 'type': 'string'},
                'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'}
            },
            'required': ['dt'],
            'type': 'object'
        }
    }
}